# SAR Despeckling and Data Compression Architecture Design

This notebook outlines the design of our SAR despeckling and compression architecture according to PEP 8 standards, with proper deterministic behavior, and revised according to the actual model architecture specifications.


## 1. Import Required Libraries


In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
import pytorch_lightning as pl
from pytorch_lightning import LightningModule

# Ensure compatibility with CompressAI
try:
    import compressai
    from compressai.models import CompressionModel
    from compressai.entropy_models import EntropyBottleneck, GaussianConditional
    from compressai.layers import GDN
    print(f"CompressAI version: {compressai.__version__}")
except ImportError:
    print("CompressAI not installed. Running in simulation mode.")
    
# Add the parent directory to the path to import from src
sys.path.append(os.path.join(os.getcwd(), '..'))

CompressAI not installed. Running in simulation mode.


## 2. Helper Functions for Network Architecture


In [ ]:
# Helper functions for convolution and transposed convolution layers
def conv(in_channels, out_channels, kernel_size=5, stride=1):
    return nn.Conv2d(
        in_channels,
        out_channels,
        kernel_size=kernel_size,
        stride=stride,
        padding=kernel_size//2
    )

def deconv(in_channels, out_channels, kernel_size=5, stride=1):
    return nn.ConvTranspose2d(
        in_channels,
        out_channels,
        kernel_size=kernel_size,
        stride=stride,
        padding=kernel_size//2,
        output_padding=stride-1
    )

# Residual block implementation
class ResidualBlock(nn.Module):
    """Residual block with skip connections."""
    
    def __init__(self, channels):
        super().__init__()
        self.conv1 = conv(channels, channels, kernel_size=5)
        self.gdn = GDN(channels)
        self.conv2 = conv(channels, channels, kernel_size=5)
        
    def forward(self, x):
        residual = x
        out = self.gdn(self.conv1(x))
        out = self.conv2(out)
        return out + residual

## 3. Corrected SAR Hyperprior Model


In [ ]:
class SARHyperprior(CompressionModel):
    """Scale Hyperprior model for SAR image despeckling and compression.
    
    This model implements the architecture described in the specifications with:
    - Residual blocks integrated after GDN layers in the analysis transform
    - 128 channels for main transform, 256 for hyperprior
    - Architecture designed for squared real/imaginary components
    """
    
    def __init__(self, N=128, M=256):
        """Initialize the SAR hyperprior model.
        
        Args:
            N: Number of channels for main transform (default: 128)
            M: Number of channels for hyperprior (default: 256)
        """
        super().__init__()
        
        self.entropy_bottleneck = EntropyBottleneck(N)
        self.gaussian_conditional = GaussianConditional(None)
        
        # Analysis transform (encoder g_a) with integrated residual blocks
        self.g_a_conv1 = conv(1, N, kernel_size=5, stride=2)
        self.g_a_gdn1 = GDN(N)
        self.g_a_res1 = ResidualBlock(N)
        
        self.g_a_conv2 = conv(N, N, kernel_size=5, stride=2)
        self.g_a_gdn2 = GDN(N)
        self.g_a_res2 = ResidualBlock(N)
        
        self.g_a_conv3 = conv(N, N, kernel_size=5, stride=2)
        self.g_a_gdn3 = GDN(N)
        self.g_a_res3 = ResidualBlock(N)
        
        self.g_a_conv4 = conv(N, N, kernel_size=5, stride=2)
        # No GDN after final layer before bottleneck
        
        # Synthesis transform (decoder g_s) with integrated residual blocks
        self.g_s_deconv1 = deconv(N, N, kernel_size=5, stride=2)
        # No GDN before first layer
        self.g_s_res1 = ResidualBlock(N)
        
        self.g_s_gdn1 = GDN(N, inverse=True)
        self.g_s_deconv2 = deconv(N, N, kernel_size=5, stride=2)
        self.g_s_res2 = ResidualBlock(N)
        
        self.g_s_gdn2 = GDN(N, inverse=True)
        self.g_s_deconv3 = deconv(N, N, kernel_size=5, stride=2)
        self.g_s_res3 = ResidualBlock(N)
        
        self.g_s_gdn3 = GDN(N, inverse=True)
        self.g_s_deconv4 = deconv(N, 1, kernel_size=5, stride=2)
        
        # Hyperprior analysis transform (h_a)
        self.h_a = nn.Sequential(
            nn.Identity(),  # abs operation happens in forward
            conv(N, M, kernel_size=3, stride=2),
            nn.ReLU(inplace=True),
            conv(M, M, kernel_size=5, stride=2),
            nn.ReLU(inplace=True),
            conv(M, M, kernel_size=5, stride=2),
        )
        
        # Hyperprior synthesis transform (h_s)
        self.h_s = nn.Sequential(
            deconv(M, M, kernel_size=5, stride=2),
            nn.ReLU(inplace=True),
            deconv(M, M, kernel_size=5, stride=2),
            nn.ReLU(inplace=True),
            deconv(M, N, kernel_size=3, stride=2),
        )
    
    def g_a(self, x):
        """Analysis transform (encoder) with integrated residual blocks."""
        x = self.g_a_gdn1(self.g_a_conv1(x))
        x = self.g_a_res1(x)
        
        x = self.g_a_gdn2(self.g_a_conv2(x))
        x = self.g_a_res2(x)
        
        x = self.g_a_gdn3(self.g_a_conv3(x))
        x = self.g_a_res3(x)
        
        return self.g_a_conv4(x)
    
    def g_s(self, x):
        """Synthesis transform (decoder) with integrated residual blocks."""
        x = self.g_s_deconv1(x)
        x = self.g_s_res1(x)
        
        x = self.g_s_gdn1(x)
        x = self.g_s_deconv2(x)
        x = self.g_s_res2(x)
        
        x = self.g_s_gdn2(x)
        x = self.g_s_deconv3(x)
        x = self.g_s_res3(x)
        
        x = self.g_s_gdn3(x)
        x = self.g_s_deconv4(x)
        
        return x
    
    def forward(self, x, training=True):
        """Forward pass through the model.
        
        Args:
            x: Input tensor (squared real or imaginary part)
            training: Whether the model is in training mode
            
        Returns:
            Dictionary with model outputs
        """
        # Apply analysis transform to get latent representation
        y = self.g_a(x)
        
        # Apply hyperprior to get scales
        z = self.h_a(torch.abs(y))
        z_hat, z_likelihoods = self.entropy_bottleneck(z)
        scales = self.h_s(z_hat)
        
        # Apply entropy coding
        y_hat, y_likelihoods = self.gaussian_conditional(y, scales)
        
        # Apply synthesis transform to reconstruct
        x_hat = self.g_s(y_hat)
        
        # Return different outputs based on training vs. testing mode
        if training:
            # During training, return reconstructed image and likelihoods
            return {
                "x_hat": x_hat, 
                "likelihoods": {"y": y_likelihoods, "z": z_likelihoods}
            }
        else:
            # During testing, also return latent representation for coding/joining
            return {
                "x_hat": x_hat,
                "y_hat": y_hat,
                "likelihoods": {"y": y_likelihoods, "z": z_likelihoods}
            }
    
    def encode(self, x):
        """Encode input to latent representation."""
        # Analysis transform
        y = self.g_a(x)
        
        # Hyperprior
        z = self.h_a(torch.abs(y))
        z_strings = self.entropy_bottleneck.compress(z)
        z_hat = self.entropy_bottleneck.decompress(z_strings, z.size()[-2:])
        
        # Get scales from hyperprior
        scales = self.h_s(z_hat)
        
        # Compress y with obtained scales
        indexes = self.gaussian_conditional.build_indexes(scales)
        y_strings = self.gaussian_conditional.compress(y, indexes)
        
        return {"strings": [y_strings, z_strings], "shape": z.size()[-2:], "y": y}
    
    def decode(self, y_hat):
        """Decode latent representation to image space."""
        return self.g_s(y_hat)

## 4. Lightning Module for SAR Despeckling and Compression


In [ ]:
class SARDDCModule(LightningModule):
    """Lightning Module for SAR Despeckling and Data Compression.
    
    This module implements the training and testing logic for joint
    despeckling and compression of SAR images using a Noise2Noise approach.
    """
    
    def __init__(
        self,
        lambda_=0.01,
        learning_rate=1e-4,
        model_kwargs=None,
    ):
        """Initialize the Lightning Module.
        
        Args:
            lambda_: Rate-distortion tradeoff parameter (default: 0.01)
            learning_rate: Learning rate for optimizer (default: 1e-4)
            model_kwargs: Additional model parameters (default: None)
        """
        super().__init__()
        
        # Save hyperparameters to be accessible via self.hparams
        self.save_hyperparameters()
        
        # Initialize model with default or provided parameters
        model_params = model_kwargs or {}
        self.model = SARHyperprior(**model_params)
        
        # Initialize RNG generator for deterministic behavior
        self._rng = torch.Generator()
        
    def on_fit_start(self):
        """Called at the beginning of fit."""
        # Set the RNG seed based on the trainer's global seed
        self._rng.manual_seed(self.trainer.global_seed)
    
    def calculate_bpp(self, likelihoods, input_shape):
        """Calculate bits per pixel."""
        num_pixels = input_shape[0] * input_shape[2] * input_shape[3]
        bpp = 0
        
        for likelihood in likelihoods.values():
            bpp += torch.sum(torch.log2(likelihood)) / (-num_pixels)
            
        return bpp
    
    def training_step(self, batch, batch_idx):
        """Training step using Noise2Noise approach."""
        # Get real and imaginary parts (squared)
        real_part_squared, imag_part_squared = batch["real_squared"], batch["imag_squared"]
        
        # Deterministic random switching of inputs/targets using seeded generator
        if torch.rand(1, generator=self._rng).item() > 0.5:
            input_data, target_data = real_part_squared, imag_part_squared
        else:
            input_data, target_data = imag_part_squared, real_part_squared
        
        # Forward pass
        output = self.model(input_data)
        x_hat = output["x_hat"]
        likelihoods = output["likelihoods"]
        
        # Calculate rate (bits per pixel)
        bpp = self.calculate_bpp(likelihoods, input_data.shape)
        
        # Calculate distortion (MSE between output and target)
        mse = torch.mean((x_hat - target_data) ** 2)
        
        # Rate-distortion loss
        loss = self.hparams.lambda_ * mse + bpp
        
        # Log metrics
        self.log("train/loss", loss)
        self.log("train/mse", mse)
        self.log("train/bpp", bpp)
        
        return loss
    
    def test_step(self, batch, batch_idx):
        """Test step with optimized processing of both real and imaginary parts."""
        # Get real and imaginary parts (squared)
        real_part_squared, imag_part_squared = batch["real_squared"], batch["imag_squared"]
        
        # Process real part directly through analysis transform
        real_y = self.model.g_a(real_part_squared)
        
        # Process imaginary part directly through analysis transform
        imag_y = self.model.g_a(imag_part_squared)
        
        # Concatenate latent representations for joint entropy coding
        y_concat = torch.cat([real_y, imag_y], dim=1)
        
        # Calculate latent representations for hyperprior
        z = self.model.h_a(torch.abs(y_concat))
        
        # Entropy coding
        z_hat, z_likelihoods = self.model.entropy_bottleneck(z)
        scales = self.model.h_s(z_hat)
        y_hat, y_likelihoods = self.model.gaussian_conditional(y_concat, scales)
        
        # Split encoded latent representation
        real_y_hat, imag_y_hat = torch.chunk(y_hat, 2, dim=1)
        
        # Decode both parts directly through synthesis transform
        real_x_hat = self.model.g_s(real_y_hat)
        imag_x_hat = self.model.g_s(imag_y_hat)
        
        # Average to get reflectivity estimate
        reflectivity = (real_x_hat + imag_x_hat) / 2
        
        # Calculate rate (bits per pixel)
        bpp = self.calculate_bpp(
            {"y": y_likelihoods, "z": z_likelihoods},
            real_part_squared.shape
        )
        
        # Log metrics (example metrics - replace with your specific metrics)
        self.log("test/bpp", bpp)
        
        # Calculate and log quality metrics (would need appropriate reference data)
        if "reference" in batch:
            mse = torch.mean((reflectivity - batch["reference"]) ** 2)
            self.log("test/mse", mse)
            
            # Calculate PSNR
            psnr = 10 * torch.log10(1.0 / mse)
            self.log("test/psnr", psnr)
        
        return {"reflectivity": reflectivity, "bpp": bpp}
    
    def configure_optimizers(self):
        """Configure optimizers."""
        return torch.optim.Adam(
            self.parameters(), 
            lr=self.hparams.learning_rate
        )

## 5. DataModule with Deterministic Behavior


In [ ]:
class SARDataModule(pl.LightningDataModule):
    """Lightning DataModule for SAR images.
    
    This module handles loading, preprocessing, and splitting of SAR data
    with deterministic behavior.
    """
    
    def __init__(
        self,
        data_dir: str,
        patch_size: int = 256,
        batch_size: int = 16,
        num_workers: int = 4,
        pin_memory: bool = True,
        **kwargs
    ):
        """Initialize the DataModule.
        
        Args:
            data_dir: Directory with SAR data
            patch_size: Size of image patches (default: 256)
            batch_size: Batch size (default: 16)
            num_workers: Number of workers for DataLoader (default: 4)
            pin_memory: Whether to pin memory (default: True)
        """
        super().__init__()
        
        # Save hyperparameters
        self.save_hyperparameters(logger=False)
        
        # Data transformations
        self.transforms = None  # Will be instantiated in setup()
        
        # Data split information
        self.data_train = None
        self.data_val = None
        self.data_test = None
    
    def prepare_data(self):
        """Data preparation (download, etc.) - runs once on the node."""
        pass  # Nothing to do here for SAR data, assumed to be downloaded
    
    def setup(self, stage=None):
        """Data setup per stage - runs on every process."""
        from torch.utils.data import random_split
        
        # Create transforms using the trainer's seed if available
        self.transforms = SARPreprocessTransform(
            patch_size=self.hparams.patch_size,
            seed=self.trainer.global_seed if hasattr(self, "trainer") else 42
        )
        
        # Seeds for deterministic behavior
        g = torch.Generator()
        g.manual_seed(self.trainer.global_seed if hasattr(self, "trainer") else 42)
        
        if stage == "fit" or stage is None:
            # Load all SAR datasets
            sar_full = SARDataset(
                self.hparams.data_dir, 
                patch_size=self.hparams.patch_size,
                transform=self.transforms,
                split="all",
                seed=self.trainer.global_seed if hasattr(self, "trainer") else 42
            )
            
            # Split dataset deterministically
            train_size = int(0.8 * len(sar_full))
            val_size = len(sar_full) - train_size
            self.data_train, self.data_val = random_split(
                sar_full, [train_size, val_size], generator=g
            )
        
        if stage == "test" or stage is None:
            self.data_test = SARDataset(
                self.hparams.data_dir,
                patch_size=self.hparams.patch_size,
                transform=self.transforms,
                split="test",
                seed=self.trainer.global_seed if hasattr(self, "trainer") else 42
            )
    
    def train_dataloader(self):
        """Create train dataloader."""
        from torch.utils.data import DataLoader
        import numpy as np
        
        return DataLoader(
            self.data_train,
            batch_size=self.hparams.batch_size,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            shuffle=True,
            # Use worker_init_fn for deterministic behavior in DataLoader workers
            worker_init_fn=lambda worker_id: np.random.seed(
                self.trainer.global_seed + worker_id
            )
        )
    
    def val_dataloader(self):
        """Create validation dataloader."""
        from torch.utils.data import DataLoader
        
        return DataLoader(
            self.data_val,
            batch_size=self.hparams.batch_size,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            shuffle=False
        )
    
    def test_dataloader(self):
        """Create test dataloader."""
        from torch.utils.data import DataLoader
        
        return DataLoader(
            self.data_test,
            batch_size=self.hparams.batch_size,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            shuffle=False
        )

## 6. SAR Dataset with Deterministic Behavior


In [ ]:
class SARDataset(torch.utils.data.Dataset):
    """Dataset for SAR SLC images with deterministic behavior.
    
    This dataset loads SAR images from CoSAR format and properly squares
    real and imaginary parts for model training.
    """
    
    def __init__(
        self, 
        data_dir: str, 
        patch_size: int = 256,
        transform=None,
        split="train",
        seed=42
    ):
        """Initialize the dataset.
        
        Args:
            data_dir: Directory with SAR data
            patch_size: Size of image patches (default: 256)
            transform: Transform to apply to samples (default: None)
            split: Data split ("train", "test", or "all") (default: "train")
            seed: Random seed for deterministic behavior (default: 42)
        """
        super().__init__()
        self.data_dir = Path(data_dir)
        self.patch_size = patch_size
        self.transform = transform
        self.split = split
        
        # List all SAR image files in CoSAR format
        self.file_list = self._get_file_list()
        
        # For deterministic behavior
        self.rng = np.random.RandomState(seed)
        
    def _get_file_list(self):
        """Get list of SAR image files based on split."""
        all_files = list(self.data_dir.glob("**/*.cos"))
        
        if not all_files:
            raise FileNotFoundError(f"No CoSAR files found in {self.data_dir}")
        
        # Sort files for deterministic behavior
        all_files.sort()
        
        if self.split == "all":
            return all_files
        
        # Deterministically split files based on filename hash
        train_files = []
        test_files = []
        
        for file_path in all_files:
            # Use filename hash for deterministic split
            file_hash = hash(str(file_path.name)) % 10
            if file_hash < 8:  # 80% for training
                train_files.append(file_path)
            else:  # 20% for testing
                test_files.append(file_path)
        
        return train_files if self.split == "train" else test_files
    
    def __len__(self):
        """Return the number of items in the dataset."""
        return len(self.file_list)
    
    def __getitem__(self, idx):
        """Get an item from the dataset."""
        # Import here to avoid circular imports
        from src.utils.sar_utils import cos2mat
        
        # Get file path
        file_path = self.file_list[idx]
        
        # Load SAR data using utility function
        sar_data = cos2mat(file_path)
        
        # Extract real and imaginary parts
        real_part = sar_data[:, :, 0]
        imag_part = sar_data[:, :, 1]
        
        # Calculate intensity (for reference/metrics)
        intensity = real_part**2 + imag_part**2
        
        # Extract random patch with deterministic behavior
        max_row = real_part.shape[0] - self.patch_size
        max_col = real_part.shape[1] - self.patch_size
        
        # Store the original item index in the RNG state to ensure
        # we get different patches for different items but the same
        # patch for the same item across epochs
        self.rng.seed(idx + 1000)  # Add offset to seed
        
        if max_row > 0 and max_col > 0:
            # Use instance-specific RNG for deterministic patch extraction
            start_row = self.rng.randint(0, max_row)
            start_col = self.rng.randint(0, max_col)
            
            real_patch = real_part[start_row:start_row+self.patch_size, 
                                start_col:start_col+self.patch_size]
            imag_patch = imag_part[start_row:start_row+self.patch_size, 
                                start_col:start_col+self.patch_size]
            intensity_patch = intensity[start_row:start_row+self.patch_size, 
                                    start_col:start_col+self.patch_size]
        else:
            # If image is too small, pad it
            real_patch = np.zeros((self.patch_size, self.patch_size))
            imag_patch = np.zeros((self.patch_size, self.patch_size))
            intensity_patch = np.zeros((self.patch_size, self.patch_size))
            
            # Copy available data
            real_patch[:real_part.shape[0], :real_part.shape[1]] = real_part
            imag_patch[:imag_part.shape[0], :imag_part.shape[1]] = imag_part
            intensity_patch[:intensity.shape[0], :intensity.shape[1]] = intensity
        
        # Convert to tensors
        real_tensor = torch.from_numpy(real_patch).float()
        imag_tensor = torch.from_numpy(imag_patch).float()
        intensity_tensor = torch.from_numpy(intensity_patch).float()
        
        # Square the real and imaginary parts as required by the model
        real_squared = real_tensor ** 2
        imag_squared = imag_tensor ** 2
        
        # Create sample
        sample = {
            "real": real_tensor,
            "imag": imag_tensor,
            "real_squared": real_squared,
            "imag_squared": imag_squared,
            "intensity": intensity_tensor,
            "file_path": str(file_path)
        }
        
        # Apply transformations if provided
        if self.transform:
            sample = self.transform(sample)
            
        return sample

## 7. SAR Preprocess Transform


In [ ]:
class SARPreprocessTransform:
    """Transform that implements the SAR preprocessing chain."""
    
    def __init__(self, patch_size=256, epsilon=1e-10, seed=42):
        """Initialize the transform.
        
        Args:
            patch_size: Size of image patches (default: 256)
            epsilon: Small constant for numerical stability (default: 1e-10)
            seed: Random seed for deterministic behavior (default: 42)
        """
        self.patch_size = patch_size
        self.epsilon = epsilon
        
        # For deterministic behavior in operations that need randomness
        self.rng = torch.Generator()
        self.rng.manual_seed(seed)
        
    def __call__(self, sample):
        """Apply the transform to a sample."""
        # Extract real and imaginary parts
        real_part = sample["real"]
        imag_part = sample["imag"]
        
        # Also process the squared versions
        real_squared = sample["real_squared"]
        imag_squared = sample["imag_squared"]
        
        # 1. Symmetrization (spectrum centering)
        real_part, imag_part = self._symmetrize(real_part, imag_part)
        
        # Recalculate squared values
        real_squared = real_part ** 2
        imag_squared = imag_part ** 2
        
        # 2. Point-like scatterer preservation (9dB threshold)
        real_part, imag_part = self._preserve_scatterers(real_part, imag_part)
        
        # Recalculate squared values
        real_squared = real_part ** 2
        imag_squared = imag_part ** 2
        
        # 3. Log transformation and normalization
        real_squared = self._log_normalize(real_squared)
        imag_squared = self._log_normalize(imag_squared)
        
        # Real and imaginary parts themselves (for reference)
        real_part = self._log_normalize(real_part)
        imag_part = self._log_normalize(imag_part)
        
        # Update sample with processed data
        updated_sample = {
            "real": real_part, 
            "imag": imag_part,
            "real_squared": real_squared,
            "imag_squared": imag_squared,
            "intensity": sample.get("intensity", None),  # Pass through if available
            "file_path": sample.get("file_path", None)   # Pass through if available
        }
        
        return updated_sample
    
    def _symmetrize(self, real, imag):
        """Apply symmetrization to ensure real and imaginary parts independence."""
        # Import here to avoid circular imports
        from src.utils.sar_utils import symetrisation_patch
        
        # Reshape for compatibility with the symetrisation_patch function
        if len(real.shape) == 2:
            real_reshaped = real.unsqueeze(0).unsqueeze(-1)
            imag_reshaped = imag.unsqueeze(0).unsqueeze(-1)
            
            # Apply symmetrization (zero Doppler centering)
            real_sym, imag_sym = symetrisation_patch(real_reshaped, imag_reshaped)
            
            # Reshape back to original dimensions
            real_out = real_sym.squeeze(0).squeeze(-1)
            imag_out = imag_sym.squeeze(0).squeeze(-1)
            return real_out, imag_out
        else:
            # Handle batched inputs
            batch_size = real.shape[0]
            real_list, imag_list = [], []
            
            for i in range(batch_size):
                r = real[i].unsqueeze(0).unsqueeze(-1)
                im = imag[i].unsqueeze(0).unsqueeze(-1)
                
                r_sym, im_sym = symetrisation_patch(r, im)
                
                real_list.append(r_sym.squeeze(0).squeeze(-1))
                imag_list.append(im_sym.squeeze(0).squeeze(-1))
                
            return torch.stack(real_list), torch.stack(imag_list)
    
    def _preserve_scatterers(self, real, imag):
        """Preserve point-like scatterers with power greater than 9dB threshold."""
        # Calculate power
        power = real**2 + imag**2
        
        # 9dB threshold in linear scale
        threshold = 10**(9/10)
        mask = power > threshold
        
        # Set equal values for real and imag parts (half the power each)
        half_sqrt_power = torch.sqrt(power[mask] / 2)
        
        # Create new tensors to avoid in-place operations
        real_out = real.clone()
        imag_out = imag.clone()
        
        # Apply the transform
        real_out[mask] = half_sqrt_power
        imag_out[mask] = half_sqrt_power
        
        return real_out, imag_out
    
    def _log_normalize(self, x):
        """Apply log transform and normalize to reduce dynamic range."""
        # Log transformation with epsilon for numerical stability
        x_log = torch.log(x + self.epsilon)
        
        # Normalize to [0, 1] range
        x_min, x_max = x_log.min(), x_log.max()
        x_norm = (x_log - x_min) / (x_max - x_min + self.epsilon)
        
        return x_norm

## 8. Hydra Configuration Example


In [ ]:
# Example Hydra configuration for SAR model
'''
# @package _global_

defaults:
  - _self_
  - data: sar.yaml
  - model: sar_hyperprior.yaml
  - callbacks: default.yaml
  - logger: wandb.yaml  # or any other logger as needed
  - trainer: default.yaml
  - paths: default.yaml

# Seed for reproducibility
seed: 42
determinism: True

# Training parameters
train: True
test: True
ckpt_path: null

# Metric for optimization
optimized_metric: "val/loss"

# Tags for wandb
tags: ["sar", "hyperprior", "despeckling", "compression"]
'''

# Example model configuration (would be in configs/model/sar_hyperprior.yaml)
'''
_target_: src.models.sar_ddc_module.SARDDCModule

lambda_: 0.01  # Rate-distortion tradeoff parameter
learning_rate: 1e-4

model_kwargs:
  N: 128  # Number of channels for main transform
  M: 256  # Number of channels for hyperprior
'''

# Example data configuration (would be in configs/data/sar.yaml)
'''
_target_: src.data.sar_datamodule.SARDataModule

data_dir: ${paths.data_dir}/sar_data
patch_size: 256
batch_size: 16
num_workers: 4
pin_memory: True
'''

## 9. Conclusion and Key Design Points


The architecture design presented here addresses all the requested modifications:

1. **PEP 8 Naming Conventions** - Used clear, descriptive names with proper casing (snake_case for methods/variables, CamelCase for classes)

2. **Deterministic Behavior** - Implemented throughout:

   - Used `torch.Generator()` with controlled seeds
   - Set seeds based on the global trainer seed
   - Ensured deterministic behavior in dataset splitting, patch extraction, and random operations

3. **Hydra Configuration Integration**:

   - Using `self.save_hyperparameters()` to store and access parameters
   - Provided example configuration files with proper structure
   - Allowed for hyperparameter tuning via Hydra

4. **Corrected Model Architecture**:

   - Integrated residual blocks after GDN layers in the analysis transform
   - Used 128 channels for main transforms and 256 for hyperprior
   - Organized the architecture with proper layer flow

5. **Using Squared Real/Imaginary Parts**:

   - Added explicit squaring of real and imaginary parts in dataset
   - Created dedicated `real_squared` and `imag_squared` tensors for model input

6. **Optimized Test Step**:
   - Directly called model components rather than the full forward pass
   - Separated analysis, entropy coding, and synthesis steps
   - Improved efficiency by avoiding redundant calculations

This design is ready to be implemented in the actual codebase, following the structure and patterns seen in the template.
